# EquiRelief — Week 1: Environment Setup & Data

**Goal**: Mount Drive, install all dependencies, download datasets, write shared config/utils, and build the 180-sample multilingual test set.

**Deliverables after this notebook:**
- `notebooks/config.py` — shared constants for all 4 notebooks
- `notebooks/utils.py` — shared helper functions
- `data/raw/` — all datasets downloaded
- `data/test_set/hinglish_test.json` — 60 Hinglish labelled samples
- `data/test_set/tanglish_test.json` — 60 Tanglish labelled samples
- `data/test_set/mixed_test.json` — 60 English/Hindi/Tamil samples
- `data/test_set/combined_180.json` — full merged test set

---


## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

Mounted at /content/drive
Drive mounted.


## Cell 2 — Create folder structure

In [2]:
import os, pathlib

BASE = '/content/drive/MyDrive/Equi-Relief'

DIRS = [
    'data/raw/humaid',
    'data/raw/crisisnlp',
    'data/raw/l3cube',
    'data/raw/fire2021',
    'data/raw/wikiannn',
    'data/raw/trec_is',
    'data/raw/kerala_floods',
    'data/raw/chennai_floods',
    'data/processed',
    'data/test_set',
    'models/hf_cache',
    'models/checkpoints',
    'outputs/demand_vectors',
    'outputs/plots',
    'outputs/results',
]

for d in DIRS:
    pathlib.Path(f'{BASE}/{d}').mkdir(parents=True, exist_ok=True)

print('Folder structure created:')
for d in DIRS:
    print(f'  {BASE}/{d}')

Folder structure created:
  /content/drive/MyDrive/Equi-Relief/data/raw/humaid
  /content/drive/MyDrive/Equi-Relief/data/raw/crisisnlp
  /content/drive/MyDrive/Equi-Relief/data/raw/l3cube
  /content/drive/MyDrive/Equi-Relief/data/raw/fire2021
  /content/drive/MyDrive/Equi-Relief/data/raw/wikiannn
  /content/drive/MyDrive/Equi-Relief/data/raw/trec_is
  /content/drive/MyDrive/Equi-Relief/data/raw/kerala_floods
  /content/drive/MyDrive/Equi-Relief/data/raw/chennai_floods
  /content/drive/MyDrive/Equi-Relief/data/processed
  /content/drive/MyDrive/Equi-Relief/data/test_set
  /content/drive/MyDrive/Equi-Relief/models/hf_cache
  /content/drive/MyDrive/Equi-Relief/models/checkpoints
  /content/drive/MyDrive/Equi-Relief/outputs/demand_vectors
  /content/drive/MyDrive/Equi-Relief/outputs/plots
  /content/drive/MyDrive/Equi-Relief/outputs/results


## Cell 3 — Point HuggingFace cache to Drive

> **Critical**: This ensures the ~5.2 GB of models download once and persist across Colab session restarts. Run this cell before any `transformers` import.

In [3]:
import os

os.environ['HF_HOME']             = f'{BASE}/models/hf_cache'
os.environ['TRANSFORMERS_CACHE']  = f'{BASE}/models/hf_cache'
os.environ['HF_DATASETS_CACHE']   = f'{BASE}/models/hf_cache/datasets'
os.environ['SENTENCE_TRANSFORMERS_HOME'] = f'{BASE}/models/hf_cache/sentence_transformers'

print('HF cache paths set:')
for k in ['HF_HOME','TRANSFORMERS_CACHE','HF_DATASETS_CACHE','SENTENCE_TRANSFORMERS_HOME']:
    print(f'  {k} = {os.environ[k]}')

HF cache paths set:
  HF_HOME = /content/drive/MyDrive/Equi-Relief/models/hf_cache
  TRANSFORMERS_CACHE = /content/drive/MyDrive/Equi-Relief/models/hf_cache
  HF_DATASETS_CACHE = /content/drive/MyDrive/Equi-Relief/models/hf_cache/datasets
  SENTENCE_TRANSFORMERS_HOME = /content/drive/MyDrive/Equi-Relief/models/hf_cache/sentence_transformers


## Cell 4 — Install dependencies

Takes ~3 minutes on first run. Restart runtime when prompted if Colab asks.

In [ ]:
import os, shutil, sys

if os.path.exists("/content/clean_lib"):
    shutil.rmtree("/content/clean_lib")

# Uninstall everything including peft this time
!pip uninstall -y transformers huggingface_hub accelerate datasets numpy peft

# Install the Golden Trio + a compatible peft
!pip install --no-cache-dir \
    "numpy==1.26.4" \
    "transformers==4.40.2" \
    "accelerate==0.30.0" \
    "huggingface_hub==0.23.0" \
    "datasets==2.19.0" \
    "sentence-transformers==3.0.0" \
    "peft==0.11.1"

print("\n--- INSTALL COMPLETE ---")
print("CRITICAL: You MUST go to 'Runtime' -> 'Restart Session' now.")

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: huggingface_hub 1.7.1
Uninstalling huggingface_hub-1.7.1:
  Successfully uninstalled huggingface_hub-1.7.1
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 148.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9


--- INSTALL COMPLETE ---
CRITICAL: You MUST go to 'Runtime' -> 'Restart Session' now.


In [4]:
import numpy as np
import transformers
import accelerate
from transformers import Trainer

print(f"NumPy: {np.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Accelerate: {accelerate.__version__}")
print("✅ SUCCESS: The circular import is gone!")

NumPy: 2.0.2
Transformers: 5.0.0
Accelerate: 1.13.0
✅ SUCCESS: The circular import is gone!


## Cell 5 — Write `config.py`

All four notebooks import this. Edit constants here and they propagate everywhere.

In [5]:
config_code = '''
# EquiRelief — shared configuration
# Imported by all four weekly notebooks.

import os

# ── Paths ──────────────────────────────────────────────────────────────────
BASE       = "/content/drive/MyDrive/Equi-Relief"
DATA_RAW   = f"{BASE}/data/raw"
DATA_PROC  = f"{BASE}/data/processed"
TEST_SET   = f"{BASE}/data/test_set"
HF_CACHE   = f"{BASE}/models/hf_cache"
CKPT_DIR   = f"{BASE}/models/checkpoints"
OUT_DEMAND = f"{BASE}/outputs/demand_vectors"
OUT_PLOTS  = f"{BASE}/outputs/plots"
OUT_RESULTS= f"{BASE}/outputs/results"

# ── HuggingFace cache (must be set before any transformers import) ─────────
os.environ["HF_HOME"]                    = HF_CACHE
os.environ["TRANSFORMERS_CACHE"]         = HF_CACHE
os.environ["HF_DATASETS_CACHE"]          = f"{HF_CACHE}/datasets"
os.environ["SENTENCE_TRANSFORMERS_HOME"] = f"{HF_CACHE}/sentence_transformers"

# ── Languages & regions ───────────────────────────────────────────────────
LANGUAGES = ["en", "hi", "ta", "hinglish", "tanglish"]

REGIONS = ["north", "south", "east", "west", "central"]
N_REGIONS = len(REGIONS)

RESOURCE_TYPES = ["food", "water", "medicine"]

# ── Model identifiers ─────────────────────────────────────────────────────
MODELS = {
    "mbert"    : "bert-base-multilingual-cased",
    "indicbert": "ai4bharat/indic-bert",
    "labse"    : "sentence-transformers/LaBSE",
    "ner"      : "Davlan/bert-base-multilingual-cased-ner-hrl",
    "zero_shot": "facebook/bart-large-mnli",
}

# Approximate sizes for reference
MODEL_SIZES_GB = {
    "mbert"    : 0.68,
    "indicbert": 0.47,
    "labse"    : 1.87,
    "ner"      : 0.68,
    "zero_shot": 1.52,
}

# ── RL hyperparameters ────────────────────────────────────────────────────
REWARD = dict(
    alpha = 1.0,   # efficiency weight
    lam   = 0.5,   # fairness penalty weight (lambda)
    beta  = 0.3,   # urgency bonus weight
    delta = 0.1,   # delay penalty weight
)

RL = dict(
    gamma        = 0.99,    # discount factor
    lr           = 1e-4,    # learning rate
    batch_size   = 64,
    buffer_size  = 50_000,  # replay buffer capacity
    target_update= 500,     # steps between target-network sync
    n_step       = 3,       # n-step TD
    epsilon_start= 1.0,
    epsilon_end  = 0.05,
    epsilon_decay= 10_000,  # steps
    n_episodes   = 2_000,
    ckpt_every   = 100,     # save checkpoint every N episodes
)

# ── NLP pipeline settings ─────────────────────────────────────────────────
NLP = dict(
    max_length       = 128,
    urgency_threshold= 0.6,   # probability cutoff for urgent label
    dbscan_eps       = 0.3,   # LaBSE cosine distance for dedup
    dbscan_min_samples = 2,
)

# ── Urgency labels (zero-shot candidate labels) ───────────────────────────
URGENCY_LABELS = ["urgent disaster relief needed", "general disaster update"]

# ── Seed ─────────────────────────────────────────────────────────────────
SEED = 42
'''

config_path = f'{BASE}/notebooks/config.py'
with open(config_path, 'w') as f:
    f.write(config_code.strip())

print(f'config.py written to {config_path}')

config.py written to /content/drive/MyDrive/Equi-Relief/notebooks/config.py


## Cell 6 — Write `utils.py`

In [6]:
utils_path = '/content/drive/MyDrive/Equi-Relief/notebooks/utils.py'

utils_content = r'''# EquiRelief — shared utilities
import json, re, os, random
import numpy as np
from pathlib import Path


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def load_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path: str, indent: int = 2):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=indent)
    print(f"Saved -> {path}")


def load_jsonl(path: str) -> list:
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def save_jsonl(records: list, path: str):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"Saved {len(records)} records -> {path}")


def normalise_text(text: str) -> str:
    import unicodedata
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#(\w+)", r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalise_hindi(text: str) -> str:
    try:
        from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
        normalizer = IndicNormalizerFactory().get_normalizer("hi")
        return normalizer.normalize(text)
    except Exception:
        return text


def normalise_tamil(text: str) -> str:
    try:
        from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
        normalizer = IndicNormalizerFactory().get_normalizer("ta")
        return normalizer.normalize(text)
    except Exception:
        return text


def dispatch_normalise(text: str, lang: str) -> str:
    if lang == "hi":
        return normalise_hindi(normalise_text(text))
    elif lang == "ta":
        return normalise_tamil(normalise_text(text))
    else:
        return normalise_text(text)


def empty_demand_vector(regions: list) -> dict:
    return {
        region: {
            "need": {"food": 0, "water": 0, "medicine": 0},
            "urgency": 0.0,
            "message_count": 0,
            "location_mentions": [],
            "disaster_types": [],
        }
        for region in regions
    }


def merge_demand(base: dict, new_entry: dict) -> dict:
    r = new_entry.get("region", "central")
    if r not in base:
        r = "central"
    rt = new_entry.get("resource_type", "food")
    if rt in base[r]["need"]:
        base[r]["need"][rt] += 1
    base[r]["urgency"] = max(base[r]["urgency"], new_entry.get("urgency", 0.0))
    base[r]["message_count"] += 1
    if new_entry.get("location"):
        base[r]["location_mentions"].append(new_entry["location"])
    if new_entry.get("disaster_type"):
        base[r]["disaster_types"].append(new_entry["disaster_type"])
    return base


def gini(arr) -> float:
    arr = np.array(arr, dtype=float)
    if arr.sum() == 0:
        return 0.0
    arr = np.sort(arr)
    n = len(arr)
    index = np.arange(1, n + 1)
    return (2 * (index * arr).sum()) / (n * arr.sum()) - (n + 1) / n


def service_ratio_variance(delivered: dict, need: dict) -> float:
    ratios = []
    for region in need:
        n = need[region]
        d = delivered.get(region, 0)
        ratios.append(d / n if n > 0 else 0.0)
    return float(np.var(ratios))


def print_section(title: str):
    print("\n" + "=" * 60)
    print(f"  {title}")
    print("=" * 60)
'''

with open(utils_path, 'w', encoding='utf-8') as f:
    f.write(utils_content)

print('✓ utils.py rewritten cleanly')

# Verify no syntax errors
import ast
with open(utils_path, 'r') as f:
    source = f.read()
try:
    ast.parse(source)
    print('✓ Syntax check passed — now retry: import config, utils')
except SyntaxError as e:
    print(f'✗ Still broken at line {e.lineno}: {e.msg}')

✓ utils.py rewritten cleanly
✓ Syntax check passed — now retry: import config, utils


## Cell 7 — Add `notebooks/` to sys.path

Run this once here, and copy it as the first code cell in every future notebook so `import config` works.

In [7]:
import sys
import os

BASE = '/content/drive/MyDrive/Equi-Relief'
notebooks_path = f'{BASE}/notebooks'
config_file_path = f'{notebooks_path}/config.py'

print(f"[DEBUG] Expected config.py path: {config_file_path}")
print(f"[DEBUG] Does config.py exist? {os.path.exists(config_file_path)}")

if notebooks_path not in sys.path:
    sys.path.insert(0, notebooks_path)
print(f"[DEBUG] sys.path after modification: {sys.path}")

import config, utils
utils.set_seed(config.SEED)
print('config and utils imported successfully.')
print(f'Regions : {config.REGIONS}')
print(f'Languages: {config.LANGUAGES}')

[DEBUG] Expected config.py path: /content/drive/MyDrive/Equi-Relief/notebooks/config.py
[DEBUG] Does config.py exist? True
[DEBUG] sys.path after modification: ['/content/drive/MyDrive/Equi-Relief/notebooks', '/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython']
config and utils imported successfully.
Regions : ['north', 'south', 'east', 'west', 'central']
Languages: ['en', 'hi', 'ta', 'hinglish', 'tanglish']


## Cell 8 — Download HumAID dataset

HumAID is the primary urgency / disaster-event labelled dataset (English). We use it for training the urgency classifier and for transfer to other languages.

In [ ]:
from datasets import load_dataset
import pandas as pd

print('Downloading HumAID...')
humaid = load_dataset('disaster_response_messages', trust_remote_code=True)

# Fallback: manual download via HuggingFace hub
# humaid = load_dataset('isebire/humaid_2022', trust_remote_code=True)

humaid_path = f'{config.DATA_RAW}/humaid'
humaid.save_to_disk(humaid_path)

# Quick inspection
df_train = humaid['train'].to_pandas()
print(f'HumAID train rows : {len(df_train)}')
print(f'Columns           : {list(df_train.columns)}')
print(f'Label distribution:')
label_cols = ['related', 'request', 'aid_related', 'medical_help', 'water',
              'food', 'shelter', 'medical_products', 'search_and_rescue']
print(f'Label distributions (sample of key columns):')
for col in label_cols:
    counts = df_train[col].value_counts().to_dict()
    print(f'  {col:<22} {counts}')
df_train.head(3)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/21046 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2629 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2573 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/21046 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2629 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2573 [00:00<?, ? examples/s]

HumAID train rows : 21046
Columns           : ['split', 'message', 'original', 'genre', 'related', 'PII', 'request', 'offer', 'aid_related', 'medical_help', 'medical_products', 'search_and_rescue', 'security', 'military', 'child_alone', 'water', 'food', 'shelter', 'clothing', 'money', 'missing_people', 'refugees', 'death', 'other_aid', 'infrastructure_related', 'transport', 'buildings', 'electricity', 'tools', 'hospitals', 'shops', 'aid_centers', 'other_infrastructure', 'weather_related', 'floods', 'storm', 'fire', 'earthquake', 'cold', 'other_weather', 'direct_report']
Label distribution:
Label distributions (sample of key columns):
  related                {1: 15795, 0: 5083, 2: 168}
  request                {0: 17486, 1: 3560}
  aid_related            {0: 12361, 1: 8685}
  medical_help           {0: 19392, 1: 1654}
  water                  {0: 19725, 1: 1321}
  food                   {0: 18717, 1: 2329}
  shelter                {0: 19168, 1: 1878}
  medical_products       {0: 19975,

,split,message,original,genre,related,PII,request,offer,aid_related,medical_help,...,aid_centers,other_infrastructure,weather_related,floods,storm,fire,earthquake,cold,other_weather,direct_report
0,train,Weather update - a cold front from Cuba that c...,Un front froid se retrouve sur Cuba ce matin. ...,direct,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,train,Is the Hurricane over or is it not over,Cyclone nan fini osinon li pa fini,direct,1,0,0,0,1,0,...,0,0,1,0,1,0,0,0,0,0
2,train,"says: west side of Haiti, rest of the country ...",facade ouest d Haiti et le reste du pays aujou...,direct,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Cell 10 — Download WikiANN (multilingual NER)

WikiANN provides NER labels for English, Hindi, and Tamil. Used to fine-tune mBERT for cross-lingual NER.

In [ ]:
from datasets import load_dataset

wikiann_langs = ['en', 'hi', 'ta']
wikiann_data = {}

for lang in wikiann_langs:
    print(f'Downloading WikiANN [{lang}]...')
    ds = load_dataset('wikiann', lang, trust_remote_code=True)
    wikiann_data[lang] = ds
    ds.save_to_disk(f'{config.DATA_RAW}/wikiannn/{lang}')
    print(f'  train: {len(ds["train"])} rows')

print('WikiANN download complete.')

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/10000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/10000 [00:00<?, ? examples/s]

...

Generating train split:   0%|          | 0/15000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/15000 [00:00<?, ? examples/s]

  train: 15000 rows
WikiANN download complete.


## Cell 11 — Download L3Cube HingCorpus (Hinglish language detection)

In [ ]:
from datasets import load_dataset

print('Downloading L3Cube HingCorpus...')
try:
    hing = load_dataset('l3cube-pune/hing-lid', trust_remote_code=True)
    hing.save_to_disk(f'{config.DATA_RAW}/l3cube')
    df_hing = hing['train'].to_pandas()
    print(f'L3Cube rows: {len(df_hing)}')
    print(df_hing['label'].value_counts().to_string())
except Exception as e:
    print(f'Download failed: {e}')
    print('Fallback: https://github.com/l3cube-pune/hindi-nlp  (hing-lid)')

Download failed: Dataset 'l3cube-pune/hing-lid' doesn't exist on the Hub or cannot be accessed. If the dataset is private or gated, make sure to log in with `huggingface-cli login` or visit the dataset page at https://huggingface.co/datasets/l3cube-pune/hing-lid to ask for access.
Fallback: https://github.com/l3cube-pune/hindi-nlp  (hing-lid)


In [8]:
from datasets import load_dataset
import os

fire_path = f'{config.DATA_RAW}/fire2021'
os.makedirs(fire_path, exist_ok=True)

print('Downloading FIRE 2021 Tanglish dataset from HuggingFace...')
ds_fire = load_dataset('community-datasets/tamilmixsentiment', trust_remote_code=True)
ds_fire.save_to_disk(fire_path)

df_fire = ds_fire['train'].to_pandas()
print(f'✓ Loaded {len(df_fire):,} rows')
print(f'Columns : {list(df_fire.columns)}')
print(f'\nLabel distribution:')
print(df_fire['label'].value_counts().to_string())

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'community-datasets/tamilmixsentiment' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'community-datasets/tamilmixsentiment' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/507k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/60.3k [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/3149 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/11335 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1260 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3149 [00:00<?, ? examples/s]

✓ Loaded 11,335 rows
Columns : ['text', 'label']

Label distribution:
label
0    7627
1    1448
2    1283
3     609
4     368


## Cell 12 — Download TREC-IS 2020 (urgency / information type)

Used alongside HumAID for the urgency classifier. Covers a wider variety of disaster events.

In [ ]:
# TREC-IS is not on HuggingFace — download instructions
import os

trec_path = f'{config.DATA_RAW}/trec_is'
print('TREC-IS 2020 is not available via HuggingFace datasets.')
print('Download steps:')
print('  1. Go to: http://dcs.gla.ac.uk/~richardm/TREC_IS/')
print('  2. Register and download 2020-A and 2020-B tracks')
print(f'  3. Place .json files in: {trec_path}')
print()
print('For now, we will create a small synthetic TREC-IS-style sample for pipeline testing:')

import json
synthetic_trec = [
    {"text": "People trapped in building, need immediate rescue!",    "label": "urgent",     "event": "earthquake"},
    {"text": "Roads flooded, cannot pass through district 4.",        "label": "urgent",     "event": "flood"},
    {"text": "Volunteers needed at relief camp near stadium.",        "label": "non_urgent", "event": "flood"},
    {"text": "Food supply running out at northern shelter.",          "label": "urgent",     "event": "flood"},
    {"text": "Power restored to eastern sector.",                    "label": "non_urgent", "event": "general"},
    {"text": "Missing person: elderly man last seen near river bank.","label": "urgent",     "event": "flood"},
    {"text": "Medical camp set up at community hall.",               "label": "non_urgent", "event": "general"},
    {"text": "Fire spreading near residential area, evacuate now!",   "label": "urgent",     "event": "fire"},
]

with open(f'{trec_path}/synthetic_sample.json', 'w') as f:
    json.dump(synthetic_trec, f, indent=2)

print(f'Synthetic TREC-IS sample saved ({len(synthetic_trec)} records).')

TREC-IS 2020 is not available via HuggingFace datasets.
Download steps:
  1. Go to: http://dcs.gla.ac.uk/~richardm/TREC_IS/
  2. Register and download 2020-A and 2020-B tracks
  3. Place .json files in: /content/drive/MyDrive/Equi-Relief/data/raw/trec_is

For now, we will create a small synthetic TREC-IS-style sample for pipeline testing:
Synthetic TREC-IS sample saved (8 records).


## Cell 13 — Download Kerala Floods 2018 & Chennai Floods 2015 data

These real disaster corpora are the full pipeline test cases.

In [ ]:
# Kerala and Chennai Floods datasets
# Primary source: CrisisNLP repository
# https://crisisnlp.qcri.org/

print('Kerala Floods 2018 and Chennai Floods 2015:')
print('  Source: https://crisisnlp.qcri.org/')
print(f'  Place CSV files in: {config.DATA_RAW}/kerala_floods/')
print(f'  Place CSV files in: {config.DATA_RAW}/chennai_floods/')
print()

# Create representative synthetic versions for Week 2 pipeline testing
import json
import os # Import os module to use os.makedirs

kerala_synthetic = [
    {"id": "kl001", "text": "Water level rising in Aluva, need boats immediately",                                   "lang": "en",      "district": "Ernakulam"},
    {"id": "kl002", "text": "ആലുവയിൽ വെള്ളം കയറി, ഉടൻ ഭക്ഷണം ആവശ്യമുണ്ട്",                                     "lang": "ml",      "district": "Ernakulam"},
    {"id": "kl003", "text": "Chalakudy me paani bahut zyada hai, please help karo",                                  "lang": "hinglish", "district": "Thrissur"},
    {"id": "kl004", "text": "Medicine needed at Chengannur relief camp",                                            "lang": "en",      "district": "Alappuzha"},
    {"id": "kl005", "text": "Pathanamthitta district mein log phanse hue hain, rescue chahiye",                     "lang": "hinglish", "district": "Pathanamthitta"},
    {"id": "kl006", "text": "Wayanad hospitals overcrowded, need medical supplies",                                  "lang": "en",      "district": "Wayanad"},
]

chennai_synthetic = [
    {"id": "ch001", "text": "Velachery underwater, people need food and water",                                     "lang": "en",      "district": "Chennai"},
    {"id": "ch002", "text": "Tambaram la thanni adichirku, boat anupunga",                                          "lang": "tanglish", "district": "Kanchipuram"},
    {"id": "ch003", "text": "Anna Nagar flood situation very bad, medicine shortage",                               "lang": "en",      "district": "Chennai"},
    {"id": "ch004", "text": "Maduravoyal la mazhaya nu solra, evlo neram aagum?",                                   "lang": "tanglish", "district": "Chennai"},
    {"id": "ch005", "text": "Porur lake brimming, surrounding areas flood warning issued",                          "lang": "en",      "district": "Chennai"},
    {"id": "ch006", "text": "Cuddalore district la inga padam iruku, help pannunga",                                "lang": "tanglish", "district": "Cuddalore"},
]

# Ensure directories exist before writing files
os.makedirs(f'{config.DATA_RAW}/kerala_floods', exist_ok=True)
os.makedirs(f'{config.DATA_RAW}/chennai_floods', exist_ok=True)

with open(f'{config.DATA_RAW}/kerala_floods/synthetic_sample.json', 'w', encoding='utf-8') as f:
    json.dump(kerala_synthetic, f, ensure_ascii=False, indent=2)

with open(f'{config.DATA_RAW}/chennai_floods/synthetic_sample.json', 'w', encoding='utf-8') as f:
    json.dump(chennai_synthetic, f, ensure_ascii=False, indent=2)

print(f'Kerala synthetic sample:  {len(kerala_synthetic)} records saved.')
print(f'Chennai synthetic sample: {len(chennai_synthetic)} records saved.')

Kerala Floods 2018 and Chennai Floods 2015:
  Source: https://crisisnlp.qcri.org/
  Place CSV files in: /content/drive/MyDrive/Equi-Relief/data/raw/kerala_floods/
  Place CSV files in: /content/drive/MyDrive/Equi-Relief/data/raw/chennai_floods/

Kerala synthetic sample:  6 records saved.
Chennai synthetic sample: 6 records saved.


## Cell 14 — Build Hinglish test set (60 samples)

This is an original project contribution — no disaster-specific Hinglish dataset exists. These 60 samples are manually crafted to cover the key NLP pipeline stages: language detection, NER, resource extraction, and urgency classification.

In [ ]:
# 60 Hinglish disaster messages with full labels
# Fields: id, text, lang, urgency (0/1), resources, location, disaster_type, region

hinglish_samples = [
    # --- Urgent, food needed ---
    {"id": "hi001", "text": "Yaar flood aa gaya, khaana chahiye urgently north sector mein",                        "lang": "hinglish", "urgency": 1, "resources": ["food"],    "location": "north sector",   "disaster_type": "flood",     "region": "north"},
    {"id": "hi002", "text": "Bhaiya please khana bhejo, 2 din se kuch nahi mila east camp",                        "lang": "hinglish", "urgency": 1, "resources": ["food"],    "location": "east camp",      "disaster_type": "flood",     "region": "east"},
    {"id": "hi003", "text": "Relief camp mein khana khatam ho gaya, immediately send karo",                        "lang": "hinglish", "urgency": 1, "resources": ["food"],    "location": "relief camp",    "disaster_type": "general",   "region": "central"},
    {"id": "hi004", "text": "Bachhe bhukhe hain, central area mein ration nahi hai",                               "lang": "hinglish", "urgency": 1, "resources": ["food"],    "location": "central area",   "disaster_type": "general",   "region": "central"},
    {"id": "hi005", "text": "Village mein anaj bilkul nahi, please west zone help karo",                           "lang": "hinglish", "urgency": 1, "resources": ["food"],    "location": "west zone",      "disaster_type": "flood",     "region": "west"},
    # --- Urgent, water needed ---
    {"id": "hi006", "text": "Paani nahi hai peene ko, south district mein pipes toot gaye",                        "lang": "hinglish", "urgency": 1, "resources": ["water"],   "location": "south district", "disaster_type": "flood",     "region": "south"},
    {"id": "hi007", "text": "Drinking water supply band ho gayi, north village crisis",                            "lang": "hinglish", "urgency": 1, "resources": ["water"],   "location": "north village",  "disaster_type": "general",   "region": "north"},
    {"id": "hi008", "text": "Saaf paani bhejo bhai, east shelter mein sab log beemar ho rahe",                    "lang": "hinglish", "urgency": 1, "resources": ["water"],   "location": "east shelter",   "disaster_type": "general",   "region": "east"},
    {"id": "hi009", "text": "Flood ke baad paani contaminated ho gaya, water truck chahiye",                      "lang": "hinglish", "urgency": 1, "resources": ["water"],   "location": "central",        "disaster_type": "flood",     "region": "central"},
    {"id": "hi010", "text": "Water tanker west camp nahi aaya 3 din se, urgent hai",                              "lang": "hinglish", "urgency": 1, "resources": ["water"],   "location": "west camp",      "disaster_type": "general",   "region": "west"},
    # --- Urgent, medicine needed ---
    {"id": "hi011", "text": "Dawai khatam ho gayi north hospital mein, immediately supply karo",                   "lang": "hinglish", "urgency": 1, "resources": ["medicine"],"location": "north hospital",  "disaster_type": "general",   "region": "north"},
    {"id": "hi012", "text": "Dengue patients south camp mein, medicine nahi hai",                                 "lang": "hinglish", "urgency": 1, "resources": ["medicine"],"location": "south camp",     "disaster_type": "epidemic",  "region": "south"},
    {"id": "hi013", "text": "Injured log east area mein, first aid kit chahiye jaldi",                            "lang": "hinglish", "urgency": 1, "resources": ["medicine"],"location": "east area",      "disaster_type": "flood",     "region": "east"},
    {"id": "hi014", "text": "Ek aurat delivery ke liye hospital nahi pahunch pa rahi, medical help",              "lang": "hinglish", "urgency": 1, "resources": ["medicine"],"location": "central",        "disaster_type": "general",   "region": "central"},
    {"id": "hi015", "text": "West zone mein cholera ke cases aa rahe, dawai bhejo",                               "lang": "hinglish", "urgency": 1, "resources": ["medicine"],"location": "west zone",      "disaster_type": "epidemic",  "region": "west"},
    # --- Urgent, multiple resources ---
    {"id": "hi016", "text": "North area mein khaana, paani, dawai sab chahiye, bahut bura haal",                  "lang": "hinglish", "urgency": 1, "resources": ["food","water","medicine"], "location": "north area", "disaster_type": "flood", "region": "north"},
    {"id": "hi017", "text": "Cyclone ke baad south village mein kuch nahi, please help",                          "lang": "hinglish", "urgency": 1, "resources": ["food","water"],           "location": "south village", "disaster_type": "cyclone", "region": "south"},
    {"id": "hi018", "text": "Earthquake ke baad east district mein log fanse hain, resources chahiye",            "lang": "hinglish", "urgency": 1, "resources": ["food","medicine"],        "location": "east district",  "disaster_type": "earthquake", "region": "east"},
    {"id": "hi019", "text": "Rescue karo aur khaana do, central mein log 48 ghante se phanse",                    "lang": "hinglish", "urgency": 1, "resources": ["food"],    "location": "central",        "disaster_type": "flood",     "region": "central"},
    {"id": "hi020", "text": "West relief shelter mein sabkuch khatam, urgent send karo",                         "lang": "hinglish", "urgency": 1, "resources": ["food","water","medicine"], "location": "west shelter",  "disaster_type": "flood",     "region": "west"},
    # --- Non-urgent, informational ---
    {"id": "hi021", "text": "North camp mein volunteers aa rahe hain kal",                                        "lang": "hinglish", "urgency": 0, "resources": [],           "location": "north camp",     "disaster_type": "general",   "region": "north"},
    {"id": "hi022", "text": "South area mein roads clear ho gayi hain",                                          "lang": "hinglish", "urgency": 0, "resources": [],           "location": "south area",     "disaster_type": "flood",     "region": "south"},
    {"id": "hi023", "text": "East hospital reopen ho gaya hai, patients aa sakte hain",                           "lang": "hinglish", "urgency": 0, "resources": [],           "location": "east hospital",  "disaster_type": "general",   "region": "east"},
    {"id": "hi024", "text": "Flood level kam ho raha hai central mein",                                          "lang": "hinglish", "urgency": 0, "resources": [],           "location": "central",        "disaster_type": "flood",     "region": "central"},
    {"id": "hi025", "text": "West mein bijli aa gayi wapas, situation better ho rahi hai",                       "lang": "hinglish", "urgency": 0, "resources": [],           "location": "west",           "disaster_type": "general",   "region": "west"},
    # --- Mixed urgency ---
    {"id": "hi026", "text": "Thoda khana aa gaya par aur chahiye north shelter mein",                             "lang": "hinglish", "urgency": 0, "resources": ["food"],    "location": "north shelter",  "disaster_type": "flood",     "region": "north"},
    {"id": "hi027", "text": "South village mein paani hai but medicine ki kami hai",                              "lang": "hinglish", "urgency": 1, "resources": ["medicine"],"location": "south village",  "disaster_type": "general",   "region": "south"},
    {"id": "hi028", "text": "East mein khana pahunch gaya, paani abhi bhi nahi",                                 "lang": "hinglish", "urgency": 1, "resources": ["water"],   "location": "east",           "disaster_type": "flood",     "region": "east"},
    {"id": "hi029", "text": "Central camp mein situation improve ho rahi hai",                                   "lang": "hinglish", "urgency": 0, "resources": [],           "location": "central camp",   "disaster_type": "general",   "region": "central"},
    {"id": "hi030", "text": "West area mein relief pahunch gayi, thanks to volunteers",                          "lang": "hinglish", "urgency": 0, "resources": [],           "location": "west area",      "disaster_type": "flood",     "region": "west"},
    # --- Difficult/noisy samples (stress test) ---
    {"id": "hi031", "text": "Bhai sach mein bahut bura haal hai yahan north mein 😭",                             "lang": "hinglish", "urgency": 1, "resources": [],           "location": "north",          "disaster_type": "general",   "region": "north"},
    {"id": "hi032", "text": "plzzz khana paani dawai sab bhejo south urgent urgent!!!",                          "lang": "hinglish", "urgency": 1, "resources": ["food","water","medicine"], "location": "south", "disaster_type": "general",  "region": "south"},
    {"id": "hi033", "text": "RT @helpeast: east mein bahut logo ko paani chahiye",                               "lang": "hinglish", "urgency": 1, "resources": ["water"],   "location": "east",           "disaster_type": "flood",     "region": "east"},
    {"id": "hi034", "text": "Koi bata sakta hai central shelter ka address? jaana hai",                           "lang": "hinglish", "urgency": 0, "resources": [],           "location": "central shelter","disaster_type": "general",   "region": "central"},
    {"id": "hi035", "text": "west walo ne bola relief aa raha hai, sach hai kya?",                               "lang": "hinglish", "urgency": 0, "resources": [],           "location": "west",           "disaster_type": "general",   "region": "west"},
    {"id": "hi036", "text": "North mein 500+ log hain camp mein, khana sirf 100 logo ka",                        "lang": "hinglish", "urgency": 1, "resources": ["food"],    "location": "north camp",     "disaster_type": "flood",     "region": "north"},
    {"id": "hi037", "text": "South district hospital mein oxygen khatam, life-threatening",                      "lang": "hinglish", "urgency": 1, "resources": ["medicine"],"location": "south district", "disaster_type": "general",   "region": "south"},
    {"id": "hi038", "text": "East mein ek bhi doctor nahi, medical team bhejo please",                           "lang": "hinglish", "urgency": 1, "resources": ["medicine"],"location": "east",           "disaster_type": "general",   "region": "east"},
    {"id": "hi039", "text": "central relief camp me koi nahi aa raha, abandoned feel ho raha",                   "lang": "hinglish", "urgency": 1, "resources": ["food","water"],           "location": "central",        "disaster_type": "general",   "region": "central"},
    {"id": "hi040", "text": "West mein pani ka level badh raha hai, ghar khali karo",                           "lang": "hinglish", "urgency": 1, "resources": [],           "location": "west",           "disaster_type": "flood",     "region": "west"},
    # --- Code-mixed Devanagari + Roman (stress test) ---
    {"id": "hi041", "text": "उत्तर में flood बहुत ज़्यादा है, khaana bhejo",                                    "lang": "hinglish", "urgency": 1, "resources": ["food"],    "location": "north",          "disaster_type": "flood",     "region": "north"},
    {"id": "hi042", "text": "South mein दवाई की ज़रूरत है, please help",                                       "lang": "hinglish", "urgency": 1, "resources": ["medicine"],"location": "south",          "disaster_type": "general",   "region": "south"},
    {"id": "hi043", "text": "East camp: पानी नहीं है, water truck send karo",                                  "lang": "hinglish", "urgency": 1, "resources": ["water"],   "location": "east camp",      "disaster_type": "general",   "region": "east"},
    {"id": "hi044", "text": "Central में सब ठीक है, volunteers aa gaye",                                       "lang": "hinglish", "urgency": 0, "resources": [],           "location": "central",        "disaster_type": "general",   "region": "central"},
    {"id": "hi045", "text": "West में बाढ़ कम हुई, but khaana abhi bhi chahiye",                               "lang": "hinglish", "urgency": 1, "resources": ["food"],    "location": "west",           "disaster_type": "flood",     "region": "west"},
    # --- Borderline urgency ---
    {"id": "hi046", "text": "North shelter mein 2-3 din ka khaana hai, stock khatam hone wala",                 "lang": "hinglish", "urgency": 0, "resources": ["food"],    "location": "north shelter",  "disaster_type": "flood",     "region": "north"},
    {"id": "hi047", "text": "South mein paani aa raha hai, sab log wahan jaayein",                             "lang": "hinglish", "urgency": 0, "resources": ["water"],   "location": "south",          "disaster_type": "general",   "region": "south"},
    {"id": "hi048", "text": "East hospital mein medicines thodi kam hain, replenish karo",                     "lang": "hinglish", "urgency": 0, "resources": ["medicine"],"location": "east hospital",  "disaster_type": "general",   "region": "east"},
    {"id": "hi049", "text": "Central mein sab theek lag raha hai abhi ke liye",                               "lang": "hinglish", "urgency": 0, "resources": [],           "location": "central",        "disaster_type": "general",   "region": "central"},
    {"id": "hi050", "text": "West mein situation stable hai, koi badi zaroorat nahi",                         "lang": "hinglish", "urgency": 0, "resources": [],           "location": "west",           "disaster_type": "general",   "region": "west"},
    # --- Remaining 10 mixed ---
    {"id": "hi051", "text": "Yaar north mein bohot sare bache hain, khaana urgently chahiye",                   "lang": "hinglish", "urgency": 1, "resources": ["food"],    "location": "north",          "disaster_type": "flood",     "region": "north"},
    {"id": "hi052", "text": "South mein fire lagi, log safe jagah par hain, water needed",                    "lang": "hinglish", "urgency": 1, "resources": ["water"],   "location": "south",          "disaster_type": "fire",      "region": "south"},
    {"id": "hi053", "text": "East mein rescue teams pahunch gayi, log safe hain",                             "lang": "hinglish", "urgency": 0, "resources": [],           "location": "east",           "disaster_type": "flood",     "region": "east"},
    {"id": "hi054", "text": "Central shelter fully packed, aur jagah chahiye aur khaana",                     "lang": "hinglish", "urgency": 1, "resources": ["food"],    "location": "central shelter","disaster_type": "general",   "region": "central"},
    {"id": "hi055", "text": "West mein earthquake ke baad sab kuch toot gaya, help karo",                     "lang": "hinglish", "urgency": 1, "resources": ["food","medicine"],        "location": "west",           "disaster_type": "earthquake", "region": "west"},
    {"id": "hi056", "text": "North area clear ho gayi, logon ko wapas jaane do",                             "lang": "hinglish", "urgency": 0, "resources": [],           "location": "north area",     "disaster_type": "flood",     "region": "north"},
    {"id": "hi057", "text": "South mein gaon poora doob gaya, koi nahi sun raha SOS",                        "lang": "hinglish", "urgency": 1, "resources": ["food","water","medicine"], "location": "south village",  "disaster_type": "flood", "region": "south"},
    {"id": "hi058", "text": "East relief work chal raha hai smoothly, no major issues",                      "lang": "hinglish", "urgency": 0, "resources": [],           "location": "east",           "disaster_type": "general",   "region": "east"},
    {"id": "hi059", "text": "Central mein ek bachhe ko snake bite hua, antivenom chahiye jaldi",             "lang": "hinglish", "urgency": 1, "resources": ["medicine"],"location": "central",        "disaster_type": "general",   "region": "central"},
    {"id": "hi060", "text": "West camp mein sab normal, kal se school reopen",                               "lang": "hinglish", "urgency": 0, "resources": [],           "location": "west camp",      "disaster_type": "general",   "region": "west"},
]

import json
utils.save_json(hinglish_samples, f'{config.TEST_SET}/hinglish_test.json')

urgent_count     = sum(1 for s in hinglish_samples if s['urgency'] == 1)
non_urgent_count = sum(1 for s in hinglish_samples if s['urgency'] == 0)
print(f'\nHinglish test set: {len(hinglish_samples)} samples')
print(f'  Urgent:     {urgent_count}')
print(f'  Non-urgent: {non_urgent_count}')

Saved -> /content/drive/MyDrive/Equi-Relief/data/test_set/hinglish_test.json

Hinglish test set: 60 samples
  Urgent:     40
  Non-urgent: 20


## Cell 15 — Build Tanglish test set (60 samples)

In [ ]:
tanglish_samples = [
    # --- Urgent, food ---
    {"id": "ta001", "text": "North area la saapadu illai, ungalukku help venuma?",                                 "lang": "tanglish", "urgency": 1, "resources": ["food"],    "location": "north area",    "disaster_type": "flood",     "region": "north"},
    {"id": "ta002", "text": "Inga saapadu kudukka marandhuteengala, east camp la pasi thangala",                   "lang": "tanglish", "urgency": 1, "resources": ["food"],    "location": "east camp",     "disaster_type": "general",   "region": "east"},
    {"id": "ta003", "text": "South shelter la rice tharuma, immediately kudunga",                                  "lang": "tanglish", "urgency": 1, "resources": ["food"],    "location": "south shelter", "disaster_type": "flood",     "region": "south"},
    {"id": "ta004", "text": "Pillaikku saapadu illaama irukku central camp la",                                   "lang": "tanglish", "urgency": 1, "resources": ["food"],    "location": "central camp",  "disaster_type": "general",   "region": "central"},
    {"id": "ta005", "text": "West zone la ration varavilai, urgent ah send pannunga",                             "lang": "tanglish", "urgency": 1, "resources": ["food"],    "location": "west zone",     "disaster_type": "flood",     "region": "west"},
    # --- Urgent, water ---
    {"id": "ta006", "text": "Kudikka thanni illai north la, pipes ellam damage aayitenga",                        "lang": "tanglish", "urgency": 1, "resources": ["water"],   "location": "north",         "disaster_type": "flood",     "region": "north"},
    {"id": "ta007", "text": "South area la water supply cut aayiduchu, urgent help",                             "lang": "tanglish", "urgency": 1, "resources": ["water"],   "location": "south area",    "disaster_type": "general",   "region": "south"},
    {"id": "ta008", "text": "East camp la thanni contaminated aayiruchu, clean water anupunga",                   "lang": "tanglish", "urgency": 1, "resources": ["water"],   "location": "east camp",     "disaster_type": "flood",     "region": "east"},
    {"id": "ta009", "text": "Central la thanni lorry 3 naal varala, people suffering",                           "lang": "tanglish", "urgency": 1, "resources": ["water"],   "location": "central",       "disaster_type": "general",   "region": "central"},
    {"id": "ta010", "text": "West district la thanni problem iruku, help pannunga",                              "lang": "tanglish", "urgency": 1, "resources": ["water"],   "location": "west district", "disaster_type": "flood",     "region": "west"},
    # --- Urgent, medicine ---
    {"id": "ta011", "text": "North hospital la medicine stock tharuma, sick people waiting",                      "lang": "tanglish", "urgency": 1, "resources": ["medicine"],"location": "north hospital","disaster_type": "general",   "region": "north"},
    {"id": "ta012", "text": "South camp la fever patients irukkaanga, tablets tharuma",                          "lang": "tanglish", "urgency": 1, "resources": ["medicine"],"location": "south camp",    "disaster_type": "epidemic",  "region": "south"},
    {"id": "ta013", "text": "East la wound patients, first aid illai yaar",                                     "lang": "tanglish", "urgency": 1, "resources": ["medicine"],"location": "east",          "disaster_type": "flood",     "region": "east"},
    {"id": "ta014", "text": "Central mein delivery case, ambulance varavilai, medical help",                    "lang": "tanglish", "urgency": 1, "resources": ["medicine"],"location": "central",       "disaster_type": "general",   "region": "central"},
    {"id": "ta015", "text": "West la diarrhea spreading, ORS and medicine anupunga",                            "lang": "tanglish", "urgency": 1, "resources": ["medicine"],"location": "west",          "disaster_type": "epidemic",  "region": "west"},
    # --- Urgent, multiple resources ---
    {"id": "ta016", "text": "North la saapadu, thanni, marunthu ellame venuma, very bad situation",               "lang": "tanglish", "urgency": 1, "resources": ["food","water","medicine"], "location": "north",   "disaster_type": "flood",     "region": "north"},
    {"id": "ta017", "text": "South village flood aayiduchu, saapadu thanni tharuma",                             "lang": "tanglish", "urgency": 1, "resources": ["food","water"],           "location": "south village", "disaster_type": "flood",     "region": "south"},
    {"id": "ta018", "text": "East district earthquake, saapadu marunthu anupunga",                               "lang": "tanglish", "urgency": 1, "resources": ["food","medicine"],        "location": "east district", "disaster_type": "earthquake","region": "east"},
    {"id": "ta019", "text": "Central la 40 hours aayiduchu saapadu illama, rescue panunga",                     "lang": "tanglish", "urgency": 1, "resources": ["food"],    "location": "central",       "disaster_type": "flood",     "region": "central"},
    {"id": "ta020", "text": "West relief camp la ellame tharuma, very urgent",                                  "lang": "tanglish", "urgency": 1, "resources": ["food","water","medicine"], "location": "west camp",     "disaster_type": "flood",     "region": "west"},
    # --- Non-urgent ---
    {"id": "ta021", "text": "North camp la volunteers varaanga naalaikku",                                       "lang": "tanglish", "urgency": 0, "resources": [],           "location": "north camp",    "disaster_type": "general",   "region": "north"},
    {"id": "ta022", "text": "South la roads clear aayiduchu",                                                   "lang": "tanglish", "urgency": 0, "resources": [],           "location": "south",         "disaster_type": "flood",     "region": "south"},
    {"id": "ta023", "text": "East hospital tirupaduchu, patients varalaam",                                     "lang": "tanglish", "urgency": 0, "resources": [],           "location": "east hospital", "disaster_type": "general",   "region": "east"},
    {"id": "ta024", "text": "Central la flood level kurainju varudu",                                           "lang": "tanglish", "urgency": 0, "resources": [],           "location": "central",       "disaster_type": "flood",     "region": "central"},
    {"id": "ta025", "text": "West la current vandhuduchu, things normalising",                                  "lang": "tanglish", "urgency": 0, "resources": [],           "location": "west",          "disaster_type": "general",   "region": "west"},
    # --- Noisy / hard cases ---
    {"id": "ta026", "text": "Ayyo north la enna nadakudhuna theriyala, help pannu yaar",                         "lang": "tanglish", "urgency": 1, "resources": [],           "location": "north",         "disaster_type": "general",   "region": "north"},
    {"id": "ta027", "text": "South la situation romba mosam, plz help!!!",                                     "lang": "tanglish", "urgency": 1, "resources": ["food","water","medicine"], "location": "south",   "disaster_type": "flood",     "region": "south"},
    {"id": "ta028", "text": "RT @helpeast: East la thanni problem iruku, anupunga",                             "lang": "tanglish", "urgency": 1, "resources": ["water"],   "location": "east",          "disaster_type": "flood",     "region": "east"},
    {"id": "ta029", "text": "Central camp address sollunga, vaara try pannaren",                               "lang": "tanglish", "urgency": 0, "resources": [],           "location": "central camp",  "disaster_type": "general",   "region": "central"},
    {"id": "ta030", "text": "West la relief varudhunu ketten, unmaiya?",                                       "lang": "tanglish", "urgency": 0, "resources": [],           "location": "west",          "disaster_type": "general",   "region": "west"},
    # --- Tamil script mixed in ---
    {"id": "ta031", "text": "North la வெள்ளம் adhigama iruku, saapadu anupunga",                               "lang": "tanglish", "urgency": 1, "resources": ["food"],    "location": "north",         "disaster_type": "flood",     "region": "north"},
    {"id": "ta032", "text": "South la மருந்து venuma, please help",                                          "lang": "tanglish", "urgency": 1, "resources": ["medicine"],"location": "south",         "disaster_type": "general",   "region": "south"},
    {"id": "ta033", "text": "East camp la தண்ணீர் illai, water send pannunga",                                "lang": "tanglish", "urgency": 1, "resources": ["water"],   "location": "east camp",     "disaster_type": "general",   "region": "east"},
    {"id": "ta034", "text": "Central la நிலைமை சரியா iruku",                                               "lang": "tanglish", "urgency": 0, "resources": [],           "location": "central",       "disaster_type": "general",   "region": "central"},
    {"id": "ta035", "text": "West la வெள்ளம் kurainju, but saapadu still venuma",                            "lang": "tanglish", "urgency": 1, "resources": ["food"],    "location": "west",          "disaster_type": "flood",     "region": "west"},
    # --- Remaining 25 ---
    {"id": "ta036", "text": "North camp la 600 peru irukkaanga, 200 perukkuthaan saapadu iruku",               "lang": "tanglish", "urgency": 1, "resources": ["food"],    "location": "north camp",    "disaster_type": "flood",     "region": "north"},
    {"id": "ta037", "text": "South hospital la oxygen illai, life risk",                                      "lang": "tanglish", "urgency": 1, "resources": ["medicine"],"location": "south hospital","disaster_type": "general",   "region": "south"},
    {"id": "ta038", "text": "East la doctor illai, medical team anupunga",                                   "lang": "tanglish", "urgency": 1, "resources": ["medicine"],"location": "east",          "disaster_type": "general",   "region": "east"},
    {"id": "ta039", "text": "Central camp la kandu pidikkavillai, abandoned feel",                           "lang": "tanglish", "urgency": 1, "resources": ["food","water"],           "location": "central",       "disaster_type": "general",   "region": "central"},
    {"id": "ta040", "text": "West la thanni level erudhu, veettu vida sollunga",                            "lang": "tanglish", "urgency": 1, "resources": [],           "location": "west",          "disaster_type": "flood",     "region": "west"},
    {"id": "ta041", "text": "North la rescue team vandhuduchu, people safe",                                "lang": "tanglish", "urgency": 0, "resources": [],           "location": "north",         "disaster_type": "flood",     "region": "north"},
    {"id": "ta042", "text": "South la cyclone damage, saapadu thanni ellame venuma",                        "lang": "tanglish", "urgency": 1, "resources": ["food","water"],           "location": "south",         "disaster_type": "cyclone",   "region": "south"},
    {"id": "ta043", "text": "East relief work smooth ah nadakudu, no major issues",                        "lang": "tanglish", "urgency": 0, "resources": [],           "location": "east",          "disaster_type": "general",   "region": "east"},
    {"id": "ta044", "text": "Central la snake bite case, antivenom immediately venuma",                    "lang": "tanglish", "urgency": 1, "resources": ["medicine"],"location": "central",       "disaster_type": "general",   "region": "central"},
    {"id": "ta045", "text": "West camp la sari, naalaikku school start",                                  "lang": "tanglish", "urgency": 0, "resources": [],           "location": "west camp",     "disaster_type": "general",   "region": "west"},
    {"id": "ta046", "text": "North la 2-3 naal saapadu iruku, stock mudinji poidum",                      "lang": "tanglish", "urgency": 0, "resources": ["food"],    "location": "north",         "disaster_type": "flood",     "region": "north"},
    {"id": "ta047", "text": "South la thanni varudhu, sari thaan",                                       "lang": "tanglish", "urgency": 0, "resources": ["water"],   "location": "south",         "disaster_type": "general",   "region": "south"},
    {"id": "ta048", "text": "East hospital la medicine kuraivu, replenish pannunga",                     "lang": "tanglish", "urgency": 0, "resources": ["medicine"],"location": "east hospital", "disaster_type": "general",   "region": "east"},
    {"id": "ta049", "text": "Central la everything ok for now",                                         "lang": "tanglish", "urgency": 0, "resources": [],           "location": "central",       "disaster_type": "general",   "region": "central"},
    {"id": "ta050", "text": "West stable, no immediate needs",                                          "lang": "tanglish", "urgency": 0, "resources": [],           "location": "west",          "disaster_type": "general",   "region": "west"},
    {"id": "ta051", "text": "North la mozhiyum vandhuduchu, urgent saapadu venuma",                     "lang": "tanglish", "urgency": 1, "resources": ["food"],    "location": "north",         "disaster_type": "flood",     "region": "north"},
    {"id": "ta052", "text": "South la fire, people safe, thanni venuma",                               "lang": "tanglish", "urgency": 1, "resources": ["water"],   "location": "south",         "disaster_type": "fire",      "region": "south"},
    {"id": "ta053", "text": "East la rescue over, all people accounted for",                          "lang": "tanglish", "urgency": 0, "resources": [],           "location": "east",          "disaster_type": "flood",     "region": "east"},
    {"id": "ta054", "text": "Central la full, vera camp venuma, saapadu venuma",                      "lang": "tanglish", "urgency": 1, "resources": ["food"],    "location": "central",       "disaster_type": "general",   "region": "central"},
    {"id": "ta055", "text": "West la earthquake, everything damaged, help pannunga",                  "lang": "tanglish", "urgency": 1, "resources": ["food","medicine"],        "location": "west",          "disaster_type": "earthquake","region": "west"},
    {"id": "ta056", "text": "North area clear, people return pannikittu irukkaanga",                 "lang": "tanglish", "urgency": 0, "resources": [],           "location": "north area",    "disaster_type": "flood",     "region": "north"},
    {"id": "ta057", "text": "South village muzhukka mazhaya nu, SOS SOS",                           "lang": "tanglish", "urgency": 1, "resources": ["food","water","medicine"], "location": "south village","disaster_type": "flood",     "region": "south"},
    {"id": "ta058", "text": "East la volunteers work pannraanga, going well",                       "lang": "tanglish", "urgency": 0, "resources": [],           "location": "east",          "disaster_type": "general",   "region": "east"},
    {"id": "ta059", "text": "Central la paambu kadichiruku, marunthu illai yaar urgent",           "lang": "tanglish", "urgency": 1, "resources": ["medicine"],"location": "central",       "disaster_type": "general",   "region": "central"},
    {"id": "ta060", "text": "West camp normal, tomorrow back to school",                          "lang": "tanglish", "urgency": 0, "resources": [],           "location": "west camp",     "disaster_type": "general",   "region": "west"},
]

utils.save_json(tanglish_samples, f'{config.TEST_SET}/tanglish_test.json')

urgent_count     = sum(1 for s in tanglish_samples if s['urgency'] == 1)
non_urgent_count = sum(1 for s in tanglish_samples if s['urgency'] == 0)
print(f'\nTanglish test set: {len(tanglish_samples)} samples')
print(f'  Urgent:     {urgent_count}')
print(f'  Non-urgent: {non_urgent_count}')

Saved -> /content/drive/MyDrive/Equi-Relief/data/test_set/tanglish_test.json

Tanglish test set: 60 samples
  Urgent:     40
  Non-urgent: 20


## Cell 16 — Build English/Hindi/Tamil test set (60 samples)

In [ ]:
mixed_samples = [
    # --- English (20 samples) ---
    {"id": "en001", "text": "Food packets urgently needed at north relief camp",                                 "lang": "en", "urgency": 1, "resources": ["food"],    "location": "north relief camp", "disaster_type": "flood",   "region": "north"},
    {"id": "en002", "text": "Water supply contaminated in south district after flooding",                      "lang": "en", "urgency": 1, "resources": ["water"],   "location": "south district",    "disaster_type": "flood",   "region": "south"},
    {"id": "en003", "text": "East hospital running out of antibiotics, please send supplies",                  "lang": "en", "urgency": 1, "resources": ["medicine"],"location": "east hospital",      "disaster_type": "general", "region": "east"},
    {"id": "en004", "text": "Volunteers arrived at central camp, situation improving",                         "lang": "en", "urgency": 0, "resources": [],           "location": "central camp",      "disaster_type": "general", "region": "central"},
    {"id": "en005", "text": "West village completely submerged, 200 people need evacuation",                   "lang": "en", "urgency": 1, "resources": ["food","water"],           "location": "west village",      "disaster_type": "flood",   "region": "west"},
    {"id": "en006", "text": "Rescue boats deployed to north sector, operations underway",                     "lang": "en", "urgency": 0, "resources": [],           "location": "north sector",      "disaster_type": "flood",   "region": "north"},
    {"id": "en007", "text": "Cholera outbreak suspected in south camp, medical team needed",                  "lang": "en", "urgency": 1, "resources": ["medicine"],"location": "south camp",         "disaster_type": "epidemic","region": "south"},
    {"id": "en008", "text": "East district roads cleared, relief trucks can now enter",                       "lang": "en", "urgency": 0, "resources": [],           "location": "east district",     "disaster_type": "flood",   "region": "east"},
    {"id": "en009", "text": "Central shelter at full capacity, need additional tents and food",               "lang": "en", "urgency": 1, "resources": ["food"],    "location": "central shelter",   "disaster_type": "general", "region": "central"},
    {"id": "en010", "text": "Power restored to west area, normalcy returning",                               "lang": "en", "urgency": 0, "resources": [],           "location": "west area",         "disaster_type": "general", "region": "west"},
    {"id": "en011", "text": "SOS from north island, people stranded for 72 hours, no food water medicine",   "lang": "en", "urgency": 1, "resources": ["food","water","medicine"], "location": "north island", "disaster_type": "flood",   "region": "north"},
    {"id": "en012", "text": "South relief camp receiving donations, no immediate shortages",                 "lang": "en", "urgency": 0, "resources": [],           "location": "south camp",        "disaster_type": "general", "region": "south"},
    {"id": "en013", "text": "Landslide blocked road to east village, rescue needed",                        "lang": "en", "urgency": 1, "resources": ["food","medicine"],        "location": "east village",      "disaster_type": "landslide","region": "east"},
    {"id": "en014", "text": "Central hospital treating 50+ injured from earthquake",                        "lang": "en", "urgency": 1, "resources": ["medicine"],"location": "central hospital",   "disaster_type": "earthquake","region": "central"},
    {"id": "en015", "text": "West sector flood waters receding, cleanup operations started",                "lang": "en", "urgency": 0, "resources": [],           "location": "west sector",       "disaster_type": "flood",   "region": "west"},
    {"id": "en016", "text": "North camp children need ORS, diarrhoea cases rising",                        "lang": "en", "urgency": 1, "resources": ["medicine"],"location": "north camp",         "disaster_type": "epidemic","region": "north"},
    {"id": "en017", "text": "South district schools converted to relief centres",                          "lang": "en", "urgency": 0, "resources": [],           "location": "south district",    "disaster_type": "flood",   "region": "south"},
    {"id": "en018", "text": "East area fire contained, no injuries reported",                             "lang": "en", "urgency": 0, "resources": [],           "location": "east area",         "disaster_type": "fire",    "region": "east"},
    {"id": "en019", "text": "Drinking water scarcity critical in central zone, send tankers now",         "lang": "en", "urgency": 1, "resources": ["water"],   "location": "central zone",      "disaster_type": "general", "region": "central"},
    {"id": "en020", "text": "West coast cyclone warning lifted, all clear",                              "lang": "en", "urgency": 0, "resources": [],           "location": "west coast",        "disaster_type": "cyclone", "region": "west"},
    # --- Hindi (20 samples) ---
    {"id": "hn001", "text": "उत्तर में भोजन की तत्काल आवश्यकता है, जल्दी भेजें",                          "lang": "hi", "urgency": 1, "resources": ["food"],    "location": "north",   "disaster_type": "flood",     "region": "north"},
    {"id": "hn002", "text": "दक्षिण क्षेत्र में पीने का पानी नहीं है, तुरंत मदद चाहिए",                  "lang": "hi", "urgency": 1, "resources": ["water"],   "location": "south",   "disaster_type": "flood",     "region": "south"},
    {"id": "hn003", "text": "पूर्व अस्पताल में दवाइयां खत्म हो गई हैं",                                "lang": "hi", "urgency": 1, "resources": ["medicine"],"location": "east",    "disaster_type": "general",   "region": "east"},
    {"id": "hn004", "text": "केंद्रीय राहत शिविर में स्थिति सामान्य हो रही है",                         "lang": "hi", "urgency": 0, "resources": [],           "location": "central", "disaster_type": "general",   "region": "central"},
    {"id": "hn005", "text": "पश्चिम गाँव में बाढ़ का पानी घरों में घुस गया, मदद चाहिए",               "lang": "hi", "urgency": 1, "resources": ["food","water"], "location": "west village", "disaster_type": "flood", "region": "west"},
    {"id": "hn006", "text": "उत्तर में बचाव दल पहुंच गया है",                                          "lang": "hi", "urgency": 0, "resources": [],           "location": "north",   "disaster_type": "flood",     "region": "north"},
    {"id": "hn007", "text": "दक्षिण में हैजा फैल रहा है, चिकित्सा टीम भेजो",                         "lang": "hi", "urgency": 1, "resources": ["medicine"],"location": "south",   "disaster_type": "epidemic",  "region": "south"},
    {"id": "hn008", "text": "पूर्व जिले की सड़कें साफ हो गई हैं",                                     "lang": "hi", "urgency": 0, "resources": [],           "location": "east",    "disaster_type": "flood",     "region": "east"},
    {"id": "hn009", "text": "केंद्रीय शिविर भर गया है, और भोजन चाहिए",                               "lang": "hi", "urgency": 1, "resources": ["food"],    "location": "central", "disaster_type": "general",   "region": "central"},
    {"id": "hn010", "text": "पश्चिम में बिजली वापस आ गई है",                                        "lang": "hi", "urgency": 0, "resources": [],           "location": "west",    "disaster_type": "general",   "region": "west"},
    {"id": "hn011", "text": "उत्तर में 72 घंटे से लोग फंसे हैं, कोई मदद नहीं",                      "lang": "hi", "urgency": 1, "resources": ["food","water","medicine"], "location": "north", "disaster_type": "flood", "region": "north"},
    {"id": "hn012", "text": "दक्षिण शिविर में दान आ रहा है, कोई कमी नहीं",                         "lang": "hi", "urgency": 0, "resources": [],           "location": "south",   "disaster_type": "general",   "region": "south"},
    {"id": "hn013", "text": "पूर्व में भूस्खलन, रास्ता बंद, राशन चाहिए",                           "lang": "hi", "urgency": 1, "resources": ["food"],    "location": "east",    "disaster_type": "landslide", "region": "east"},
    {"id": "hn014", "text": "केंद्र में भूकंप से 50 से अधिक घायल",                                 "lang": "hi", "urgency": 1, "resources": ["medicine"],"location": "central", "disaster_type": "earthquake","region": "central"},
    {"id": "hn015", "text": "पश्चिम में बाढ़ का पानी कम हो रहा है",                               "lang": "hi", "urgency": 0, "resources": [],           "location": "west",    "disaster_type": "flood",     "region": "west"},
    {"id": "hn016", "text": "उत्तर शिविर में बच्चों को ओआरएस चाहिए",                             "lang": "hi", "urgency": 1, "resources": ["medicine"],"location": "north",   "disaster_type": "epidemic",  "region": "north"},
    {"id": "hn017", "text": "दक्षिण जिले के स्कूल राहत केंद्र बने",                              "lang": "hi", "urgency": 0, "resources": [],           "location": "south",   "disaster_type": "flood",     "region": "south"},
    {"id": "hn018", "text": "पूर्व में आग बुझ गई, कोई घायल नहीं",                               "lang": "hi", "urgency": 0, "resources": [],           "location": "east",    "disaster_type": "fire",      "region": "east"},
    {"id": "hn019", "text": "केंद्र में पीने का पानी बेहद जरूरी, टैंकर भेजो",                   "lang": "hi", "urgency": 1, "resources": ["water"],   "location": "central", "disaster_type": "general",   "region": "central"},
    {"id": "hn020", "text": "पश्चिम में चक्रवात की चेतावनी हटाई गई",                           "lang": "hi", "urgency": 0, "resources": [],           "location": "west",    "disaster_type": "cyclone",   "region": "west"},
    # --- Tamil (20 samples) ---
    {"id": "tm001", "text": "வடக்கில் உணவு உடனடியாக தேவை, அனுப்புங்கள்",                                   "lang": "ta", "urgency": 1, "resources": ["food"],    "location": "north",   "disaster_type": "flood",     "region": "north"},
    {"id": "tm002", "text": "தெற்கில் குடிநீர் பிரச்சனை, உடனடி உதவி வேண்டும்",                          "lang": "ta", "urgency": 1, "resources": ["water"],   "location": "south",   "disaster_type": "flood",     "region": "south"},
    {"id": "tm003", "text": "கிழக்கு மருத்துவமனையில் மருந்து இல்லை",                                   "lang": "ta", "urgency": 1, "resources": ["medicine"],"location": "east",    "disaster_type": "general",   "region": "east"},
    {"id": "tm004", "text": "மத்திய முகாமில் நிலைமை சரியாகி வருகிறது",                                 "lang": "ta", "urgency": 0, "resources": [],           "location": "central", "disaster_type": "general",   "region": "central"},
    {"id": "tm005", "text": "மேற்கு கிராமம் முழுவதும் நீரில் மூழ்கி உள்ளது",                           "lang": "ta", "urgency": 1, "resources": ["food","water"],           "location": "west village", "disaster_type": "flood",   "region": "west"},
    {"id": "tm006", "text": "வடக்கில் மீட்பு படை வந்துவிட்டது",                                      "lang": "ta", "urgency": 0, "resources": [],           "location": "north",   "disaster_type": "flood",     "region": "north"},
    {"id": "tm007", "text": "தெற்கில் காலரா பரவுகிறது, மருத்துவ குழு வேண்டும்",                      "lang": "ta", "urgency": 1, "resources": ["medicine"],"location": "south",   "disaster_type": "epidemic",  "region": "south"},
    {"id": "tm008", "text": "கிழக்கு மாவட்ட சாலைகள் சரி செய்யப்பட்டன",                              "lang": "ta", "urgency": 0, "resources": [],           "location": "east",    "disaster_type": "flood",     "region": "east"},
    {"id": "tm009", "text": "மத்திய தஞ்சாவூர் முகாம் நிரம்பி விட்டது, உணவு வேண்டும்",              "lang": "ta", "urgency": 1, "resources": ["food"],    "location": "central", "disaster_type": "general",   "region": "central"},
    {"id": "tm010", "text": "மேற்கில் மின்சாரம் வந்துவிட்டது",                                      "lang": "ta", "urgency": 0, "resources": [],           "location": "west",    "disaster_type": "general",   "region": "west"},
    {"id": "tm011", "text": "வடக்கில் 72 மணி நேரமாக மக்கள் சிக்கியுள்ளனர், SOS",                    "lang": "ta", "urgency": 1, "resources": ["food","water","medicine"], "location": "north", "disaster_type": "flood",   "region": "north"},
    {"id": "tm012", "text": "தெற்கு முகாமில் நன்கொடை வருகிறது",                                    "lang": "ta", "urgency": 0, "resources": [],           "location": "south",   "disaster_type": "general",   "region": "south"},
    {"id": "tm013", "text": "கிழக்கில் நிலச்சரிவு, சாலை மூடல், ரேஷன் வேண்டும்",                   "lang": "ta", "urgency": 1, "resources": ["food"],    "location": "east",    "disaster_type": "landslide", "region": "east"},
    {"id": "tm014", "text": "மையத்தில் நிலநடுக்கம், 50+ பேர் காயம்",                              "lang": "ta", "urgency": 1, "resources": ["medicine"],"location": "central", "disaster_type": "earthquake","region": "central"},
    {"id": "tm015", "text": "மேற்கில் வெள்ள நீர் குறைகிறது",                                       "lang": "ta", "urgency": 0, "resources": [],           "location": "west",    "disaster_type": "flood",     "region": "west"},
    {"id": "tm016", "text": "வடக்கு முகாமில் குழந்தைகளுக்கு ORS வேண்டும்",                       "lang": "ta", "urgency": 1, "resources": ["medicine"],"location": "north",   "disaster_type": "epidemic",  "region": "north"},
    {"id": "tm017", "text": "தெற்கில் பள்ளிகள் நிவாரண மையங்களாக மாற்றப்பட்டன",                  "lang": "ta", "urgency": 0, "resources": [],           "location": "south",   "disaster_type": "flood",     "region": "south"},
    {"id": "tm018", "text": "கிழக்கில் தீ அணைக்கப்பட்டது, யாரும் காயமடையவில்லை",                "lang": "ta", "urgency": 0, "resources": [],           "location": "east",    "disaster_type": "fire",      "region": "east"},
    {"id": "tm019", "text": "மையத்தில் குடிநீர் மிகவும் அவசியம், டேங்கர் அனுப்புங்கள்",        "lang": "ta", "urgency": 1, "resources": ["water"],   "location": "central", "disaster_type": "general",   "region": "central"},
    {"id": "tm020", "text": "மேற்கு கடலோரத்தில் புயல் எச்சரிக்கை நீக்கப்பட்டது",              "lang": "ta", "urgency": 0, "resources": [],           "location": "west",    "disaster_type": "cyclone",   "region": "west"},
]

utils.save_json(mixed_samples, f'{config.TEST_SET}/mixed_test.json')

for lang in ['en', 'hi', 'ta']:
    subset = [s for s in mixed_samples if s['lang'] == lang]
    urgent = sum(1 for s in subset if s['urgency'] == 1)
    print(f'{lang}: {len(subset)} samples | urgent: {urgent} | non-urgent: {len(subset)-urgent}')

Saved -> /content/drive/MyDrive/Equi-Relief/data/test_set/mixed_test.json
en: 20 samples | urgent: 11 | non-urgent: 9
hi: 20 samples | urgent: 11 | non-urgent: 9
ta: 20 samples | urgent: 11 | non-urgent: 9


## Cell 17 — Merge into combined_180.json

In [ ]:
import json

combined = hinglish_samples + tanglish_samples + mixed_samples
assert len(combined) == 180, f'Expected 180, got {len(combined)}'

utils.save_json(combined, f'{config.TEST_SET}/combined_180.json')

# Summary statistics
import collections
lang_counts     = collections.Counter(s['lang']     for s in combined)
region_counts   = collections.Counter(s['region']   for s in combined)
disaster_counts = collections.Counter(s['disaster_type'] for s in combined)
urgent_total    = sum(1 for s in combined if s['urgency'] == 1)

print(f'\nCombined test set: {len(combined)} samples')
print(f'  Urgent:     {urgent_total}')
print(f'  Non-urgent: {len(combined) - urgent_total}')
print(f'\nBy language:')
for lang, count in sorted(lang_counts.items()):
    print(f'  {lang:<12} {count}')
print(f'\nBy region:')
for region, count in sorted(region_counts.items()):
    print(f'  {region:<12} {count}')
print(f'\nBy disaster type:')
for dtype, count in sorted(disaster_counts.items(), key=lambda x: -x[1]):
    print(f'  {dtype:<15} {count}')

Saved -> /content/drive/MyDrive/Equi-Relief/data/test_set/combined_180.json

Combined test set: 180 samples
  Urgent:     113
  Non-urgent: 67

By language:
  en           20
  hi           20
  hinglish     60
  ta           20
  tanglish     60

By region:
  central      37
  east         36
  north        36
  south        35
  west         36

By disaster type:
  general         77
  flood           73
  epidemic        10
  earthquake      7
  cyclone         5
  fire            5
  landslide       3


## Cell 18 — Smoke test: import config + load test set

In [ ]:
# Reload config fresh (simulates how Week 2 will import it)
import importlib, sys

for mod in ['config', 'utils']:
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

import config, utils

# Load the combined test set
test_set = utils.load_json(f'{config.TEST_SET}/combined_180.json')
print(f'Loaded {len(test_set)} samples from combined_180.json')

# Spot-check one sample per language
spot_langs = ['hinglish', 'tanglish', 'en', 'hi', 'ta']
print('\nSpot checks:')
for lang in spot_langs:
    sample = next(s for s in test_set if s['lang'] == lang)
    print(f'  [{lang}] {sample["id"]}: "{sample["text"][:60]}..."')
    print(f'         urgency={sample["urgency"]} resources={sample["resources"]} region={sample["region"]}')

# Test utils functions
import numpy as np
test_gini = utils.gini([10, 20, 30, 40])
test_var  = utils.service_ratio_variance({'north': 5, 'south': 2}, {'north': 10, 'south': 10})
print(f'\nutils.gini([10,20,30,40])                    = {test_gini:.4f} (expected ~0.25)')
print(f'utils.service_ratio_variance(...)            = {test_var:.4f} (expected 0.0225)')

# Test empty demand vector
dv = utils.empty_demand_vector(config.REGIONS)
print(f'\nempty_demand_vector regions: {list(dv.keys())}')
print('All checks passed.')

Loaded 180 samples from combined_180.json

Spot checks:
  [hinglish] hi001: "Yaar flood aa gaya, khaana chahiye urgently north sector mei..."
         urgency=1 resources=['food'] region=north
  [tanglish] ta001: "North area la saapadu illai, ungalukku help venuma?..."
         urgency=1 resources=['food'] region=north
  [en] en001: "Food packets urgently needed at north relief camp..."
         urgency=1 resources=['food'] region=north
  [hi] hn001: "उत्तर में भोजन की तत्काल आवश्यकता है, जल्दी भेजें..."
         urgency=1 resources=['food'] region=north
  [ta] tm001: "வடக்கில் உணவு உடனடியாக தேவை, அனுப்புங்கள்..."
         urgency=1 resources=['food'] region=north

utils.gini([10,20,30,40])                    = 0.2500 (expected ~0.25)
utils.service_ratio_variance(...)            = 0.0225 (expected 0.0225)

empty_demand_vector regions: ['north', 'south', 'east', 'west', 'central']
All checks passed.


## Cell 19 — Verify Drive structure

In [9]:
import os

def tree(path, prefix='', max_depth=3, depth=0):
    if depth > max_depth:
        return
    try:
        entries = sorted(os.listdir(path))
    except PermissionError:
        return
    for i, entry in enumerate(entries):
        connector = '└── ' if i == len(entries) - 1 else '├── '
        full = os.path.join(path, entry)
        size = ''
        if os.path.isfile(full):
            sz = os.path.getsize(full)
            size = f' ({sz:,} bytes)' if sz < 1_000_000 else f' ({sz/1e6:.1f} MB)'
        print(f'{prefix}{connector}{entry}{size}')
        if os.path.isdir(full):
            extension = '    ' if i == len(entries) - 1 else '│   '
            tree(full, prefix + extension, max_depth, depth + 1)

print(f'Equi-Relief/ on Drive')
tree(BASE)

Equi-Relief/ on Drive
├── data
│   ├── processed
│   │   ├── figureeight_mapped.csv (6.3 MB)
│   │   ├── humaid_mapped.csv (13.6 MB)
│   │   └── master_clean.csv (28,368 bytes)
│   ├── raw
│   │   ├── chennai_floods
│   │   │   └── synthetic_sample.json (861 bytes)
│   │   ├── crisisnlp
│   │   │   ├── events_set1
│   │   │   └── events_set2
│   │   ├── fire2021
│   │   │   ├── dataset_dict.json (43 bytes)
│   │   │   ├── test
│   │   │   ├── train
│   │   │   └── validation
│   │   ├── humaid
│   │   │   ├── dataset_dict.json (43 bytes)
│   │   │   ├── test
│   │   │   ├── train
│   │   │   └── validation
│   │   ├── kerala_floods
│   │   │   └── synthetic_sample.json (933 bytes)
│   │   ├── l3cube
│   │   ├── trec_is
│   │   │   └── synthetic_sample.json (936 bytes)
│   │   └── wikiannn
│   │       ├── en
│   │       ├── hi
│   │       └── ta
│   └── test_set
│       ├── combined_180.json (48,415 bytes)
│       ├── hinglish_test.json (15,791 bytes)
│       ├── mixed_test.json (17,337

## Cell 20 — Week 1 summary

Everything needed for Week 2 (NLP pipeline) is now in place.

In [ ]:
utils.print_section('Week 1 complete — deliverables summary')

deliverables = [
    (f'{config.BASE}/notebooks/config.py',           'Shared config for all notebooks'),
    (f'{config.BASE}/notebooks/utils.py',            'Shared utilities for all notebooks'),
    (f'{config.TEST_SET}/hinglish_test.json',        '60 labelled Hinglish samples'),
    (f'{config.TEST_SET}/tanglish_test.json',        '60 labelled Tanglish samples'),
    (f'{config.TEST_SET}/mixed_test.json',           '60 EN + HI + TA samples'),
    (f'{config.TEST_SET}/combined_180.json',         '180-sample combined test set'),
    (f'{config.DATA_RAW}/kerala_floods/synthetic_sample.json', 'Kerala Floods synthetic'),
    (f'{config.DATA_RAW}/chennai_floods/synthetic_sample.json','Chennai Floods synthetic'),
    (f'{config.DATA_RAW}/trec_is/synthetic_sample.json',       'TREC-IS synthetic'),
]

all_ok = True
for path, description in deliverables:
    exists = os.path.exists(path)
    status = '✓' if exists else '✗ MISSING'
    if not exists:
        all_ok = False
    print(f'  {status}  {description}')
    print(f'       {path}')

print()
if all_ok:
    print('All deliverables present. Ready for Week 2 — NLP Pipeline.')
else:
    print('Some files missing — re-run the relevant cells above.')

print()
print('Next: open week2_nlp.ipynb')
print('      First cell: import sys; sys.path.insert(0, "/content/drive/MyDrive/Equi-Relief/notebooks")')
print('      Then: import config, utils')


  Week 1 complete — deliverables summary
  ✓  Shared config for all notebooks
       /content/drive/MyDrive/Equi-Relief/notebooks/config.py
  ✓  Shared utilities for all notebooks
       /content/drive/MyDrive/Equi-Relief/notebooks/utils.py
  ✓  60 labelled Hinglish samples
       /content/drive/MyDrive/Equi-Relief/data/test_set/hinglish_test.json
  ✓  60 labelled Tanglish samples
       /content/drive/MyDrive/Equi-Relief/data/test_set/tanglish_test.json
  ✓  60 EN + HI + TA samples
       /content/drive/MyDrive/Equi-Relief/data/test_set/mixed_test.json
  ✓  180-sample combined test set
       /content/drive/MyDrive/Equi-Relief/data/test_set/combined_180.json
  ✓  Kerala Floods synthetic
       /content/drive/MyDrive/Equi-Relief/data/raw/kerala_floods/synthetic_sample.json
  ✓  Chennai Floods synthetic
       /content/drive/MyDrive/Equi-Relief/data/raw/chennai_floods/synthetic_sample.json
  ✓  TREC-IS synthetic
       /content/drive/MyDrive/Equi-Relief/data/raw/trec_is/synthetic_sampl